# 09. Telemetry Module

`TelemetryModule` is the observability nucleus of an arcllm stack. It wraps the
adapter, measures every `invoke()` call, computes cost from per-1M-token
pricing baked into the provider TOML, and — new in v0.4 — routes a
`TraceRecord` to an `on_event` callback and/or a `TraceStore` for persistent,
hash-chained audit trails.

This notebook teaches telemetry by *running real telemetry against a mock
adapter*. No API keys required; events are captured into a list and inspected
field-by-field.

You will learn:

- Which event types `TelemetryModule` emits and the exact payload fields
- How phase sub-timings (`prompt_assembly_ms`, `llm_call_ms`,
  `post_processing_ms`, `total_ms`) decompose a single call
- How to wire an `on_event` callback through `load_model()` so an external
  dashboard or memory store sees every call
- How `agent_label` attributes traces to a specific agent in multi-agent setups
- How cost is computed per call and aggregated across runs
- How to set latency budgets and alarm on slow calls
- How `TraceRecord` flows from `TelemetryModule` into a `TraceStore`
  (cross-ref: notebook 14)

## 1. Setup

In [1]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

False

Pull the imports we need across the rest of the notebook. We define a
single `MockAdapter` that returns a canned `LLMResponse` and tracks how many
times it was called — every section reuses it so we can compare module
behavior side-by-side.

In [2]:
import asyncio
import logging
import time
from typing import Any
from unittest.mock import patch

from arcllm.modules.telemetry import (
    TelemetryModule,
    BudgetAccumulator,
    clear_budgets,
    clear_global_defaults,
    set_global_defaults,
)
from arcllm.registry import clear_cache, load_model
from arcllm.trace_store import TraceRecord
from arcllm.types import LLMProvider, LLMResponse, Message, Tool, Usage


class MockAdapter(LLMProvider):
    """Minimal adapter that returns canned responses without network I/O."""

    def __init__(
        self,
        name: str = "anthropic",
        model_name: str = "claude-sonnet-4-6",
        usage: Usage | None = None,
        latency_s: float = 0.0,
        stop_reason: str = "end_turn",
    ) -> None:
        # LLMProvider.name / .model_name are abstract *properties* — a plain
        # `self.name = ...` instance assignment does not satisfy them (the
        # ABC check runs against the class, before __init__ ever executes).
        # Store on private attrs and expose via the properties below.
        self._name = name
        self._model_name = model_name
        self._usage = usage or Usage(input_tokens=120, output_tokens=40, total_tokens=160)
        self._latency_s = latency_s
        self._stop_reason = stop_reason
        self.calls: list[tuple[list[Message], list[Tool] | None, dict[str, Any]]] = []

    @property
    def name(self) -> str:
        return self._name

    @property
    def model_name(self) -> str:
        return self._model_name

    async def invoke(
        self,
        messages: list[Message],
        tools: list[Tool] | None = None,
        **kwargs: Any,
    ) -> LLMResponse:
        self.calls.append((messages, tools, kwargs))
        if self._latency_s > 0:
            await asyncio.sleep(self._latency_s)
        return LLMResponse(
            content="ok",
            usage=self._usage,
            model=self.model_name,
            stop_reason=self._stop_reason,
        )

    def validate_config(self) -> bool:
        return True

    async def close(self) -> None:
        pass


# Anthropic Sonnet 4.6 pricing (matches providers/anthropic.toml)
SONNET_PRICING = {
    "cost_input_per_1m": 3.00,
    "cost_output_per_1m": 15.00,
    "cost_cache_read_per_1m": 0.30,
    "cost_cache_write_per_1m": 3.75,
}

# Reset shared registries — keeps repeated nbconvert runs deterministic
clear_budgets()
clear_global_defaults()
clear_cache()
print("setup OK")

setup OK


## 2. Why telemetry — observability, cost, latency

When you operate thousands of agents, every LLM call needs to answer three
questions for you in real time:

1. **What happened?** Who called whom, with which model, and did it succeed?
2. **What did it cost?** Tokens in/out, cache hits, dollar amount.
3. **How long did it take?** Wall-clock total *and* the sub-phases inside it.

`TelemetryModule` is the single source of truth for those three answers. It
sits inside the standard module stack:

```
Otel → Queue → Telemetry → CircuitBreaker → Audit → Security →
       Retry → Fallback → RateLimit → [Router|Adapter]
```

Telemetry's position matters: it measures wall-clock *as the agent
experiences it*, but is inside `Queue` so queue wait time is attributed to
`Queue` (where it belongs) rather than smeared into the LLM call.

Telemetry never blocks the call path — it is purely observational. If
a callback raises or a `TraceStore.append` is slow, that's recorded but the
adapter response still flows back to the caller. The minimum overhead path
when nothing is configured (no callback, no trace store, no budget) is a few
`time.monotonic()` calls and one structured log line.

## 3. Event types emitted — full list with payload shapes

`TelemetryModule` emits three observable signals per `invoke()`:

| Channel | Where it lands | Payload |
|---|---|---|
| **Structured log** | `arcllm.modules.telemetry` logger | `LLM call \| provider=… model=… duration_ms=… cost_usd=…` |
| **OTel attributes** | Active span `arcllm.telemetry` | `arcllm.telemetry.duration_ms`, `arcllm.telemetry.cost_usd` |
| **TraceRecord** | `on_event(record)` and/or `trace_store.append(record)` | Full record (see §9) |

It also emits OTel events on budget breaches: `budget_exceeded` and
`budget_alert`. Those are covered in notebook 15 alongside the queue and
circuit breaker — here we focus on the per-call telemetry signals.

Run a single call and look at all three signals at once.

In [3]:
# 1. Capture structured logs
log_records: list[logging.LogRecord] = []


class _CaptureHandler(logging.Handler):
    def emit(self, record: logging.LogRecord) -> None:
        log_records.append(record)


tel_logger = logging.getLogger("arcllm.modules.telemetry")
tel_logger.addHandler(_CaptureHandler(level=logging.INFO))
tel_logger.setLevel(logging.INFO)

# 2. Capture TraceRecord events via on_event
events: list[TraceRecord] = []


def on_event(record: TraceRecord) -> None:
    events.append(record)


# 3. Wire the module
adapter = MockAdapter()
mod = TelemetryModule(
    {**SONNET_PRICING, "on_event": on_event, "agent_label": "demo-agent"},
    adapter,
)

resp = await mod.invoke([Message(role="user", content="hi")])

print(f"adapter calls   : {len(adapter.calls)}")
print(f"log records     : {len(log_records)}")
print(f"events captured : {len(events)}")

adapter calls   : 1
log records     : 1
events captured : 1


The structured log line is the human-friendly version. Note how `None`
values (cache tokens, in this case) are dropped automatically by
`log_structured()` to keep lines tight.

In [4]:
last_log = log_records[-1].getMessage()
print(last_log)

LLM call | provider=anthropic model=claude-sonnet-4-6 duration_ms=0.000000 input_tokens=120 output_tokens=40 total_tokens=160 cost_usd=0.000960 stop_reason=end_turn


The `TraceRecord` is the machine-friendly version — a frozen Pydantic
model with every field your dashboards, billing, and audit pipelines need.

In [5]:
rec = events[-1]
print(f"trace_id        : {rec.trace_id}")
print(f"timestamp       : {rec.timestamp}")
print(f"provider/model  : {rec.provider} / {rec.model}")
print(f"agent_label     : {rec.agent_label}")
print(f"duration_ms     : {rec.duration_ms}")
print(f"cost_usd        : {rec.cost_usd:.6f}")
print(f"tokens          : in={rec.input_tokens} out={rec.output_tokens}")
print(f"stop_reason     : {rec.stop_reason}")
print(f"status          : {rec.status}")
print(f"event_type      : {rec.event_type}")
print(f"phase_timings   : {sorted(rec.phase_timings)}")

trace_id        : 29670f049ff648169620264cd9cdc153
timestamp       : 2026-07-09T11:38:31.387386+00:00
provider/model  : anthropic / claude-sonnet-4-6
agent_label     : demo-agent
duration_ms     : 0.0
cost_usd        : 0.000960
tokens          : in=120 out=40
stop_reason     : end_turn
status          : success
event_type      : llm_call
phase_timings   : ['llm_call_ms', 'post_processing_ms', 'prompt_assembly_ms', 'total_ms']


Each `TraceRecord` is hash-chained when persisted by a `TraceStore`
(notebook 14). `on_event`-only callers receive the un-chained record — useful
for ephemeral observers like UI sockets that don't need tamper-evidence.

In [6]:
# Reset the captures so later sections start clean
events.clear()
log_records.clear()

## 4. Phase timings (sub-spans within a single call)

A single `invoke()` is split into four phases:

| Phase | Boundary | What happens |
|---|---|---|
| `prompt_assembly_ms` | `t0 → t_pre` | Budget pre-check, internal kwarg stripping |
| `llm_call_ms` | `t_pre → t_llm` | The actual `inner.invoke()` (where retries, queue, adapter live) |
| `post_processing_ms` | `t_llm → t_post` | Cost calc, budget deduct, response model_copy |
| `total_ms` | `t0 → t_post` | Sum of the three above |

Phase timings are attached to the `TraceRecord.phase_timings` dict so
dashboards can break down where time is going.

In [7]:
# Patch time.monotonic so we get deterministic phase timings.
# 4 calls in invoke(): t0, t_pre, t_llm, t_post.
fake_times = iter([0.000, 0.005, 0.500, 0.510])

with patch("arcllm.modules.telemetry.time.monotonic", lambda: next(fake_times)):
    adapter = MockAdapter()
    mod = TelemetryModule(
        {**SONNET_PRICING, "on_event": on_event},
        adapter,
    )
    await mod.invoke([Message(role="user", content="hi")])

rec = events[-1]
for k, v in rec.phase_timings.items():
    print(f"{k:>22}: {v} ms")

    prompt_assembly_ms: 5.0 ms
           llm_call_ms: 495.0 ms
    post_processing_ms: 10.0 ms
              total_ms: 510.0 ms


IOStream.flush timed out


The split lets you ask "did the LLM get slow, or did our pre-flight
get slow?" without instrumenting your own code. In a real run with all
modules enabled, `llm_call_ms` includes everything Telemetry wraps —
CircuitBreaker, Audit, Security, Retry, Fallback, RateLimit, and the actual
HTTP roundtrip. To break those out further, use OTel spans (notebook 11).

In [8]:
# Phase timings serialize through the TraceRecord so they survive
# round-trip to JSONL on disk.
import json

dumped = json.dumps(rec.model_dump(), default=str, indent=2)
print(dumped[:600])

{
  "trace_id": "74f3b7439d8b4b4d91df5e555d3addb3",
  "timestamp": "2026-07-09T11:38:31.397787+00:00",
  "provider": "anthropic",
  "model": "claude-sonnet-4-6",
  "agent_label": null,
  "agent_did": null,
  "budget_scope": null,
  "request_body": {
    "messages": [
      {
        "role": "user",
        "content": "hi"
      }
    ],
    "tools": null
  },
  "response_body": {
    "content": "ok",
    "tool_calls": [],
    "stop_reason": "end_turn"
  },
  "classification": "unclassified",
  "encryption": null,
  "lineage": null,
  "duration_ms": 510.0,
  "cost_usd": 0.00096,
  "input_tokens


## 5. `on_event` callback wiring (NEW in v0.4)

Before v0.4 you had to subclass `TelemetryModule` or scrape logs to react to
calls. v0.4 added a clean callback channel: pass `on_event=...` to
`load_model()` and your function receives a `TraceRecord` after every call.

The callback fires *after* the response is returned to the caller — it cannot
delay the call. It's invoked inside the `invoke()` task, so it's safe to do
small synchronous work (append to a list, increment a counter, push to an
asyncio Queue). For heavy work, hand off to a background task.

The full set of `on_event`-related kwargs on `load_model()` is documented in
notebook 04 (`arcllm/04-agentic-loop.ipynb`). Here we focus on what
TelemetryModule does with it.

In [9]:
# load_model() threads on_event into TelemetryModule for you.
clear_cache()

events.clear()


def collector(record: TraceRecord) -> None:
    events.append(record)


# Set a fake API key so the adapter constructor doesn't trip on missing env.
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key")

# Build a model with telemetry enabled and on_event wired.
model = load_model(
    "anthropic",
    telemetry=True,
    on_event=collector,
    agent_label="agent-7",
)

# Walk the wrapper stack to confirm Telemetry is on top.
layer = model
chain = []
while hasattr(layer, "_inner"):
    chain.append(type(layer).__name__)
    layer = layer._inner
chain.append(type(layer).__name__)
print(" → ".join(chain))

QueueModule → TelemetryModule → AuditModule → RetryModule → FallbackModule → RateLimitModule → AnthropicAdapter


Notice how `load_model()` injects pricing (from `anthropic.toml`)
*and* the `on_event` callback into the same `TelemetryModule` instance. That
single object is responsible for emitting both signals.

Now invoke against a swapped-in mock adapter so we don't hit the network. The
common pattern in tests is to construct `TelemetryModule` directly rather
than going through `load_model()` — but for completeness, here's a way to
exercise the wired model end-to-end with a mock.

In [10]:
# Replace the innermost adapter with our mock so invoke() works offline.
mock = MockAdapter()

# The innermost layer is the AnthropicAdapter — replace its `invoke` and
# patch the response usage so cost is computed.
innermost = model
while hasattr(innermost, "_inner"):
    innermost = innermost._inner


# Monkeypatch invoke for the duration of the call. (Tests usually inject a
# MockAdapter directly into TelemetryModule; this just shows the seam.)
async def _fake_invoke(messages, tools=None, **kwargs):
    return await mock.invoke(messages, tools, **kwargs)


_orig_invoke = innermost.invoke
innermost.invoke = _fake_invoke
try:
    await model.invoke([Message(role="user", content="hello")])
finally:
    innermost.invoke = _orig_invoke

print(f"events captured: {len(events)}")
print(f"agent_label   : {events[-1].agent_label}")
print(f"cost_usd      : {events[-1].cost_usd:.6f}")

events captured: 1
agent_label   : agent-7
cost_usd      : 0.000960


**Why callbacks fire outside locks.** `TelemetryModule._emit_trace()`
calls `on_event(record)` *after* releasing any internal locks. A slow or
buggy callback cannot block other in-flight calls. The same applies to
`trace_store.append()`.

In [11]:
# Demonstrate that exceptions in on_event do NOT propagate to the caller.
def buggy(record: TraceRecord) -> None:
    raise RuntimeError("downstream is on fire")


bad_mod = TelemetryModule({**SONNET_PRICING, "on_event": buggy}, MockAdapter())

# Telemetry will surface the callback error — that's by design (fail loud
# during development). For production, wrap your callback in try/except.
try:
    await bad_mod.invoke([Message(role="user", content="hi")])
    print("invoke succeeded (callback not called)")
except RuntimeError as e:
    print(f"callback raised: {e}")

callback raised: downstream is on fire


Buggy callbacks DO propagate by default — wrap your callback if you
need fail-soft behavior. The contract: *Telemetry is observational, but you
own your callback's reliability.*

## 6. `agent_label` attribution (NEW in v0.4)

In a single-agent app, every trace is "yours". In a team of agents,
`agent_label` is what tells dashboards/billing/audit which agent fired each
call. The label flows through the trace record as a string and through the
JSONL store's query filter (`/api/traces?agent=…`).

Conventions:
- Use stable agent identifiers, not display names: `"scap_isso"` not `"My
  ISSO Agent"`
- Subcomponents append with `/`: `"scap_isso/memory"`. The trace store treats
  `agent=scap_isso` as a prefix match, so it returns sub-component traces
  too.

In [12]:
# Three agents, one shared callback. Telemetry attributes each call.
events.clear()


def shared_collector(rec: TraceRecord) -> None:
    events.append(rec)


for label in ["agent-alpha", "agent-beta", "agent-alpha/memory"]:
    mod = TelemetryModule(
        {**SONNET_PRICING, "on_event": shared_collector, "agent_label": label},
        MockAdapter(),
    )
    await mod.invoke([Message(role="user", content=f"hi from {label}")])

for r in events:
    print(f"{r.agent_label:<24} cost=${r.cost_usd:.6f} dur={r.duration_ms}ms")

agent-alpha              cost=$0.000960 dur=0.0ms
agent-beta               cost=$0.000960 dur=0.0ms
agent-alpha/memory       cost=$0.000960 dur=0.0ms


Cost aggregation by label is now a one-liner. This is the building
block for billing dashboards and per-agent budget enforcement.

In [13]:
from collections import defaultdict

totals: dict[str, float] = defaultdict(float)
for rec in events:
    # Treat 'alpha/memory' as belonging to 'alpha' for top-level rollup
    root = (rec.agent_label or "unknown").split("/")[0]
    totals[root] += rec.cost_usd

for label, usd in sorted(totals.items()):
    print(f"{label:<16} ${usd:.6f}")

agent-alpha      $0.001920
agent-beta       $0.000960


`agent_did` is the cryptographic counterpart to `agent_label` — the
DID is the identity, the label is the human-readable shorthand. Both flow
through the trace record. arctrust (notebook arctrust/01) covers DID
construction; here we just confirm the field plumbing.

In [14]:
mod = TelemetryModule(
    {
        **SONNET_PRICING,
        "agent_label": "agent-alpha",
        "agent_did": "did:arc:agent-alpha:abcdef",
        "on_event": shared_collector,
    },
    MockAdapter(),
)

events.clear()
await mod.invoke([Message(role="user", content="hi")])
print(f"label : {events[-1].agent_label}")
print(f"did   : {events[-1].agent_did}")

label : agent-alpha
did   : did:arc:agent-alpha:abcdef


**Global defaults for multi-agent setups.** Calling
`set_global_defaults(on_event=…, trace_store=…, agent_did=…)` once at app
startup wires every subsequent `TelemetryModule` instance to the same sinks.
Useful when an arcui process owns the dashboard socket and every agent
created within it should publish to it.

In [15]:
# clean slate, then wire global defaults
clear_global_defaults()
events.clear()

set_global_defaults(on_event=shared_collector, agent_did="did:arc:operator:root")

# Subsequently-created modules inherit the defaults unless overridden.
mod_a = TelemetryModule({**SONNET_PRICING, "agent_label": "alpha"}, MockAdapter())
mod_b = TelemetryModule({**SONNET_PRICING, "agent_label": "beta"}, MockAdapter())

await mod_a.invoke([Message(role="user", content="hi")])
await mod_b.invoke([Message(role="user", content="hi")])

for r in events:
    print(f"{r.agent_label:<8} did={r.agent_did}")

clear_global_defaults()

alpha    did=did:arc:operator:root
beta     did=did:arc:operator:root


## 7. Cost tracking — per-call USD, aggregate

Cost is computed as:

```
cost_usd = (input_tokens  × cost_input_per_1m  / 1_000_000)
         + (output_tokens × cost_output_per_1m / 1_000_000)
         + (cache_read_tokens  × cost_cache_read_per_1m  / 1_000_000  if any)
         + (cache_write_tokens × cost_cache_write_per_1m / 1_000_000  if any)
```

Pricing comes from `arcllm/providers/<provider>.toml`. `load_model()` injects
it via `setdefault()` so the user can still override per-call by passing
`telemetry={"cost_input_per_1m": …}`.

In [16]:
# Hand-construct the math so you can verify against the module
adapter = MockAdapter(usage=Usage(input_tokens=1000, output_tokens=500, total_tokens=1500))
mod = TelemetryModule(SONNET_PRICING, adapter)

cost = mod._calculate_cost(adapter._usage)
expected = (1000 * 3.00 + 500 * 15.00) / 1_000_000
print(f"computed : ${cost:.6f}")
print(f"expected : ${expected:.6f}")
assert abs(cost - expected) < 1e-12

computed : $0.010500
expected : $0.010500


In [17]:
# Cache tokens add their own line items (only when present)
adapter_cached = MockAdapter(
    usage=Usage(
        input_tokens=1000,
        output_tokens=500,
        total_tokens=1500,
        cache_read_tokens=8000,  # huge cache hit
        cache_write_tokens=200,
    )
)
mod_cached = TelemetryModule(SONNET_PRICING, adapter_cached)
cost_with_cache = mod_cached._calculate_cost(adapter_cached._usage)
print(f"with cache    : ${cost_with_cache:.6f}")

# Cache reads are 10x cheaper than fresh input — the savings are visible
adapter_nocache = MockAdapter(usage=Usage(input_tokens=9000, output_tokens=500, total_tokens=9500))
mod_nocache = TelemetryModule(SONNET_PRICING, adapter_nocache)
print(f"without cache : ${mod_nocache._calculate_cost(adapter_nocache._usage):.6f}")

with cache    : $0.013650
without cache : $0.034500


**Aggregate cost by collecting `TraceRecord.cost_usd` over a session.**
This is exactly what arcui's billing panel does — it subscribes via
`on_event`, sums by `agent_label`, and re-renders.

In [18]:
events.clear()


def collect(rec: TraceRecord) -> None:
    events.append(rec)


usages = [
    Usage(input_tokens=120, output_tokens=80, total_tokens=200),
    Usage(input_tokens=400, output_tokens=200, total_tokens=600),
    Usage(input_tokens=80, output_tokens=400, total_tokens=480),
    Usage(
        input_tokens=200,
        output_tokens=300,
        total_tokens=500,
        cache_read_tokens=4000,
    ),
]

for usage in usages:
    mod = TelemetryModule(
        {**SONNET_PRICING, "on_event": collect, "agent_label": "billing-demo"},
        MockAdapter(usage=usage),
    )
    await mod.invoke([Message(role="user", content="hi")])

session_cost = sum(r.cost_usd for r in events)
session_in = sum(r.input_tokens for r in events)
session_out = sum(r.output_tokens for r in events)
print(f"calls       : {len(events)}")
print(f"input toks  : {session_in:,}")
print(f"output toks : {session_out:,}")
print(f"session $   : ${session_cost:.6f}")

calls       : 4
input toks  : 800
output toks : 980
session $   : $0.018300


**Cost on the response object.** The cost is also written back onto
the response via `response.model_copy(update={"cost_usd": cost})`. Callers
who want it without subscribing can read `response.cost_usd`.

In [19]:
adapter = MockAdapter()
mod = TelemetryModule(SONNET_PRICING, adapter)

resp = await mod.invoke([Message(role="user", content="hi")])
print(f"resp.cost_usd     : ${resp.cost_usd:.6f}")
print(f"resp.usage.total  : {resp.usage.total_tokens}")

resp.cost_usd     : $0.000960
resp.usage.total  : 160


**Free pricing → zero cost.** Self-hosted endpoints (Ollama, vLLM)
have all-zero pricing in their TOMLs. The cost field is still present and
serializable — just always 0.0.

In [20]:
mod_free = TelemetryModule({}, MockAdapter())
print(
    f"free cost: ${mod_free._calculate_cost(Usage(input_tokens=1_000_000, output_tokens=1_000_000, total_tokens=2_000_000)):.6f}"
)

free cost: $0.000000


## 8. Latency budgets — alarms / warnings

Telemetry doesn't ship a built-in latency-budget primitive — its job is to
*emit*, not to *enforce*. But the emission contract is rich enough that you
build budgets in three lines on top of `on_event`:

```python
def latency_alarm(rec):
    if rec.duration_ms > 5_000:
        my_alerter.fire(
            f"{rec.agent_label}: {rec.model} took {rec.duration_ms}ms"
        )
```

The same idea works for cost (alarm if `cost_usd > 0.10`), token cap (alarm
if `input_tokens > 100_000`), or any composite of the fields.

In [21]:
# A toy alarm/warn pipeline. In real systems this is a Slack hook or
# Prometheus counter increment.
alarms: list[str] = []
warnings: list[str] = []

WARN_MS = 200.0
ALARM_MS = 800.0


def budget_watch(rec: TraceRecord) -> None:
    if rec.duration_ms >= ALARM_MS:
        alarms.append(f"ALARM {rec.agent_label}: {rec.duration_ms}ms (>= {ALARM_MS}ms)")
    elif rec.duration_ms >= WARN_MS:
        warnings.append(f"WARN  {rec.agent_label}: {rec.duration_ms}ms (>= {WARN_MS}ms)")


# Simulate three calls with different durations via patched time.monotonic.
fake = iter(
    [
        # call 1: 50ms (silent)
        0.0,
        0.005,
        0.050,
        0.055,
        # call 2: 300ms (warn)
        0.0,
        0.005,
        0.300,
        0.305,
        # call 3: 1200ms (alarm)
        0.0,
        0.010,
        1.190,
        1.200,
    ]
)

with patch("arcllm.modules.telemetry.time.monotonic", lambda: next(fake)):
    for label in ["fast", "slow", "very-slow"]:
        mod = TelemetryModule(
            {**SONNET_PRICING, "on_event": budget_watch, "agent_label": label},
            MockAdapter(),
        )
        await mod.invoke([Message(role="user", content="hi")])

print("alarms  :", alarms)
print("warnings:", warnings)

alarms  : ['ALARM very-slow: 1200.0ms (>= 800.0ms)']
warnings: ['WARN  slow: 305.0ms (>= 200.0ms)']


**OTel-native budgets.** The same fields are also OTel span
attributes (`arcllm.telemetry.duration_ms`, `arcllm.telemetry.cost_usd`),
so a Prometheus exporter or Grafana alert can do the same job without an
in-process callback. Pick one — both is double-counting.

**Spend budgets.** For *spend* (not latency) you want
`TelemetryModule`'s built-in budget enforcement: pass `monthly_limit_usd`,
`daily_limit_usd`, `per_call_max_usd`, and a `budget_scope`, and the module
will pre-check (block before the call) and post-deduct (after a successful
call). That deeper topic is covered in notebook 15 alongside the queue and
circuit breaker.

In [22]:
# Quick taste of budget config — the limits raise BudgetError on breach.
from arcllm.exceptions import ArcLLMBudgetError

clear_budgets()

mod_budget = TelemetryModule(
    {
        **SONNET_PRICING,
        "monthly_limit_usd": 100.00,
        "daily_limit_usd": 5.00,
        "per_call_max_usd": 0.001,  # tiny — first call will breach pre-flight
        "enforcement": "block",
        "budget_scope": "demo:notebook9",
        "default_max_tokens": 4096,
    },
    MockAdapter(),
)

try:
    await mod_budget.invoke([Message(role="user", content="hi")])
except ArcLLMBudgetError as e:
    print(f"BudgetError raised: limit_type={e.limit_type} estimated=${e.estimated_usd:.6f}")

clear_budgets()

BudgetError raised: limit_type=per_call estimated=$0.061440


For the full budget API (warn vs block enforcement, calendar resets,
per-scope accumulators, alert thresholds), see notebook 15. The plumbing
above is enough to know where it lives.

## 9. TraceRecord integration (cross-ref `arcllm/14-trace-store.ipynb`)

`on_event` is great for ephemeral observers. For *persistent* audit trails,
pass a `TraceStore` to `load_model(trace_store=…)` and Telemetry will
`await trace_store.append(record)` after every call.

The store hash-chains every record. Notebook 14 covers the chain semantics
(`compute_hash`, `verify_chain`, daily rotation, tamper detection). Here we
just close the loop: confirm Telemetry produces a record the store accepts.

In [23]:
# In-memory trace store that mirrors the JSONLTraceStore append contract.
class InMemoryTraceStore:
    def __init__(self) -> None:
        self.records: list[TraceRecord] = []
        self._last_hash = "0" * 64

    async def append(self, record: TraceRecord) -> None:
        hashed = record.with_hash(self._last_hash)
        self._last_hash = hashed.record_hash
        self.records.append(hashed)


store = InMemoryTraceStore()

mod = TelemetryModule(
    {**SONNET_PRICING, "trace_store": store, "agent_label": "demo"},
    MockAdapter(),
)

for _ in range(3):
    await mod.invoke([Message(role="user", content="hi")])

print(f"records persisted: {len(store.records)}")
print(f"last hash        : {store._last_hash[:16]}…")
for r in store.records:
    print(f"  prev={r.prev_hash[:8]}… self={r.record_hash[:8]}… cost=${r.cost_usd:.6f}")

records persisted: 3
last hash        : ef43c9a5265c4d8e…
  prev=00000000… self=78000d0c… cost=$0.000960
  prev=78000d0c… self=badfbe60… cost=$0.000960
  prev=badfbe60… self=ef43c9a5… cost=$0.000960


The chain is intact: each record's `prev_hash` matches the prior
record's `record_hash`. Notebook 14 takes this further — verifying the chain,
detecting tampering, and rotating files daily on disk.

In [24]:
# Verify the chain manually
prev = "0" * 64
chain_ok = True
for r in store.records:
    if r.prev_hash != prev:
        chain_ok = False
        break
    prev = r.record_hash
print("chain OK:", chain_ok)

chain OK: True


**Both `on_event` AND `trace_store` at the same time.** Telemetry
fires both — the callback first, then the store. They are independent;
either, both, or neither is fine. arcui dashboards typically wire both:
`on_event` for live UI updates and `trace_store` for the durable log.

In [25]:
events.clear()
store2 = InMemoryTraceStore()


def live_observer(rec: TraceRecord) -> None:
    events.append(rec)


mod = TelemetryModule(
    {
        **SONNET_PRICING,
        "on_event": live_observer,
        "trace_store": store2,
        "agent_label": "ui+store",
    },
    MockAdapter(),
)

await mod.invoke([Message(role="user", content="hi")])
print(f"on_event saw   : {len(events)} records")
print(f"trace_store has: {len(store2.records)} records")

on_event saw   : 1 records
trace_store has: 1 records


**Raw bodies on/off.** By default Telemetry includes the request and
response bodies in the trace record (`store_raw_bodies=True`). Set it to
False to keep records lean — useful for high-volume scenarios or when bodies
contain regulated content that lives in a separate, signed audit pipeline.

In [26]:
mod_no_bodies = TelemetryModule(
    {
        **SONNET_PRICING,
        "on_event": live_observer,
        "agent_label": "no-bodies",
        "store_raw_bodies": False,
    },
    MockAdapter(),
)

events.clear()
await mod_no_bodies.invoke([Message(role="user", content="secret")])
rec = events[-1]
print(f"request_body  : {rec.request_body}")
print(f"response_body : {rec.response_body}")

request_body  : None
response_body : None


**`set_global_defaults` for multi-instance setups.** When arcui boots
it calls `set_global_defaults(on_event=…, trace_store=…, agent_did=…)`. Every
TelemetryModule instance built afterwards (by any agent in the process)
inherits those defaults unless the per-module config explicitly overrides
them. We saw this in §6 — here's the same idea with a trace store wired in
globally.

In [27]:
clear_global_defaults()

global_store = InMemoryTraceStore()
set_global_defaults(trace_store=global_store)

# Modules built AFTER set_global_defaults pick up the global store
mod_x = TelemetryModule({**SONNET_PRICING, "agent_label": "x"}, MockAdapter())
mod_y = TelemetryModule({**SONNET_PRICING, "agent_label": "y"}, MockAdapter())

await mod_x.invoke([Message(role="user", content="hi")])
await mod_y.invoke([Message(role="user", content="hi")])

print(f"global store records: {len(global_store.records)}")
for r in global_store.records:
    print(f"  agent={r.agent_label} cost=${r.cost_usd:.6f}")

clear_global_defaults()

global store records: 2
  agent=x cost=$0.000960
  agent=y cost=$0.000960


## 10. Summary

`TelemetryModule` is the observability nucleus of an arcllm stack. v0.4
elevated it from a structured-log emitter to the entry point for live
callbacks, persistent trace stores, multi-agent attribution, and budget
enforcement — all without breaking the v0.3 contract.

**Public API exercised in this notebook:**
- `arcllm.modules.telemetry.TelemetryModule`
- `arcllm.modules.telemetry.set_global_defaults`
- `arcllm.modules.telemetry.clear_global_defaults`
- `arcllm.modules.telemetry.clear_budgets`
- `arcllm.modules.telemetry.BudgetAccumulator` (referenced)
- `arcllm.trace_store.TraceRecord`
- `arcllm.registry.load_model(on_event=…, trace_store=…, agent_label=…, telemetry=…)`
- `arcllm.registry.clear_cache`
- `TelemetryModule._calculate_cost` (illustrative)
- Phase timings: `prompt_assembly_ms`, `llm_call_ms`, `post_processing_ms`,
  `total_ms`
- TraceRecord fields: `trace_id`, `timestamp`, `provider`, `model`,
  `agent_label`, `agent_did`, `duration_ms`, `cost_usd`, `input_tokens`,
  `output_tokens`, `total_tokens`, `cache_read_tokens`, `cache_write_tokens`,
  `stop_reason`, `status`, `phase_timings`, `prev_hash`, `record_hash`,
  `event_type`, `request_body`, `response_body`

**Three signals per call:**
1. Structured log line (`arcllm.modules.telemetry`)
2. OTel span attributes (`arcllm.telemetry.duration_ms`,
   `arcllm.telemetry.cost_usd`)
3. `TraceRecord` dispatched to `on_event` and/or `trace_store`

**Threading via `load_model()`** is the production path:

```python
model = load_model(
    "anthropic",
    telemetry=True,
    on_event=ui_socket.broadcast,
    trace_store=jsonl_store,
    agent_label="agent-7",
)
```

**Where to go next:**
- `arcllm/04-agentic-loop.ipynb` — full `load_model()` kwarg surface and
  module stack ordering
- `arcllm/14-trace-store.ipynb` — JSONLTraceStore, hash-chain verification,
  daily rotation, tamper detection
- `arcllm/15-queue-circuit-breaker.ipynb` — budget enforcement, queue
  backpressure, circuit-breaker state machine
- `arcllm/11-otel-export.ipynb` — exporting Telemetry's spans/attributes to
  Prometheus/Tempo/Jaeger
- `arctrust/04-audit-sinks.ipynb` — emitting `AuditEvent` from
  TelemetryModule via `arctrust.audit.emit`